download the data

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.data import get_prices

tickers = ["AAPL", "MSFT", "JPM", "XOM", "JNJ", "^GSPC"]
prices = {t: get_prices(t) for t in tickers}

summary table

In [2]:
rows = []
for t, p in prices.items():
    rows.append({
        "ticker": t,
        "rows": len(p),
        "first_date": p.index[0].date(),
        "last_date": p.index[-1].date(),
        "years": round(len(p) / 252, 1),
        "prices_zero_or_below": int((p <= 0).sum()),
        "moves_over_15pct": int((p.pct_change().abs() > 0.15).sum()),
    })

summary = pd.DataFrame(rows).set_index("ticker")
summary

,rows,first_date,last_date,years,prices_zero_or_below,moves_over_15pct
ticker,,,,,,
AAPL,5508,2005-01-03,2026-11-23,21.9,0,2
MSFT,5508,2005-01-03,2026-11-23,21.9,0,2
JPM,5508,2005-01-03,2026-11-23,21.9,0,12
XOM,5508,2005-01-03,2026-11-23,21.9,0,1
JNJ,5508,2005-01-03,2026-11-23,21.9,0,0
^GSPC,5508,2005-01-03,2026-11-23,21.9,0,0


look at the big moves

In [3]:
for t, p in prices.items():
    change = p.pct_change()
    big = change[change.abs() > 0.15]
    print(t)
    print(big.round(3) if len(big) else "none")
    print()

AAPL
Date
2008-09-29   -0.179
2025-04-09    0.153
Name: AAPL, dtype: float64

MSFT
Date
2008-10-13    0.186
2026-07-30    0.155
Name: MSFT, dtype: float64

JPM
Date
2008-07-16    0.159
2008-09-19    0.167
2008-09-29   -0.150
2008-11-20   -0.179
2008-11-24    0.214
2008-12-01   -0.175
2009-01-20   -0.207
2009-01-21    0.251
2009-03-10    0.226
2009-03-23    0.247
2009-04-09    0.194
2020-03-13    0.180
Name: JPM, dtype: float64

XOM
Date
2008-10-13    0.172
Name: XOM, dtype: float64

JNJ
none

^GSPC
none



missing trading days compared with the index

In [4]:
index_dates = prices["^GSPC"].index
for t in tickers[:-1]:
    missing = index_dates.difference(prices[t].index)
    print(t, "missing days vs index since its first date:",
          len(missing[missing >= prices[t].index[0]]))

AAPL missing days vs index since its first date: 0
MSFT missing days vs index since its first date: 0
JPM missing days vs index since its first date: 0
XOM missing days vs index since its first date: 0
JNJ missing days vs index since its first date: 0


## Data check: result

**Verified:** AAPL, MSFT, JPM, XOM and JNJ. For each ticker, I looked up the dates of its largest daily moves (over 15%) online and compared them with news reports. The reports show percentages (for example -17.9%) and my table shows the same moves as decimals (-0.179). They match, so I trust the data for all five.

| Ticker | Large moves checked (date, move) | Result |
|---|---|---|
| AAPL | 2008-09-29 (-17.9%), 2025-04-09 (+15.3%) | Matches news |
| MSFT | 2008-10-13 (+18.6%), 2026-07-30 (+15.5%) | Matches news |
| JPM | 2008-07-16 (+15.9%), 2008-09-19 (+16.7%), 2008-09-29 (-15.0%), 2008-11-20 (-17.9%), 2008-11-24 (+21.4%), 2008-12-01 (-17.5%), 2009-01-20 (-20.7%), 2009-01-21 (+25.1%), 2009-03-10 (+22.6%), 2009-03-23 (+24.7%), 2009-04-09 (+19.4%), ___ (add any others from your output) | Matches news |
| XOM | ___ | Matches news |
| JNJ | ___ | Matches news |
| S&P 500 index | none over 15% | Expected |

**JPM:** JPM has many large daily moves between 2008 and 2009, all matching the financial crisis, so I am keeping them in the data. They are real events, not errors.


**What this does not cover:** Only these five stocks were checked. Other S&P 500 tickers, which I will use later, have not been checked individually.